Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [68]:
NO_RECORDS = 150

In [69]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [70]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [71]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [72]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [73]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320', 'title': 'Multiresolution scanning and measurement of coherence', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/540928320</identifier>\n    <datestamp>2021-09-14T06:03:32.145Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-540928320</identifier><creators><creator><creatorName nameType="P

In [74]:
record_titles

['Multiresolution scanning and measurement of coherence',
 'Projects in protein crystallography of GSY',
 'The Evolution of Crystallization and Phase Separation Equilibrium in Multi-Component Systems for Spray-Coated Organic Solar Cells',
 'HERCULES Practicals',
 'CryoEM BAG for Ile-de-France',
 'London Cancer BAG comprising the Institute of Cancer Research; the Crick Institute and the University of Sussex',
 'Quantification and control of 5f versus 6d contributions to spin and orbital angular momenta in early actinide paramagnetism.',
 'CRYO-EM STUDIES OF PROTEINS RELATED TO PATHOGENESIS OF TRANSMISSIBLE AND NON-TRANSMISSIBLE DISEASES, CELL DIVISION, SIGNALLING AND CHROMATIN',
 'Testing of new bending electrocatalytic cell',
 'Structure-activity relationships of homogeneous Pd species in high-pressure diene-CO2 reactions forming carbon-fixating monomers',
 'Austrian Crystallographic Diffraction Consortium',
 'Probing Spatial Correlations of Charge Density Waves and Structural Variatio

In [75]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2051950665</identifier>\n    <datestamp>2025-04-22T06:00:23.963Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2051950665</identifier><creators><creator><creatorName nameType="Personal">Shuxian XIONG</creatorName><givenName>Shuxian</givenName><familyName>Xiong</familyName></creator><creator><creatorName nameType="Perso

In [76]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-2051950665'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [77]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [78]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'Particles, Nuclei and Fields', 'BM05', 'IH-MA-175']


In [79]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [80]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/161963533', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/895887653', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313092290', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1945024757', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2002919516', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1471181319', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2264618071', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2318800294', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2227639430', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/536305320', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1582583397', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1220329319', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/15531713

In [81]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [82]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Bergamasco, Enrico', 'name': 'Enrico', 'surname': 'Bergamasco', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0009-0001-1477-1290'}, 'provenance': None}}, {'fullName': 'Carrillo Larrea, Juan Manuel', 'name': 'Juan Manuel', 'surname': 'Carrillo Larrea', 'rank': 2, 'pid': None}, {'fullName': 'Eshesheva, Aidana', 'name': 'Aidana', 'surname': 'Eshesheva', 'rank': 3, 'pid': None}, {'fullName': 'Hubley, Austin', 'name': 'Austin', 'surname': 'Hubley', 'rank': 4, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-6326-1951'}, 'provenance': None}}, {'fullName': 'Mckenzie, Alexandre', 'name': 'Alexandre', 'surname': 'Mckenzie', 'rank': 5, 'pid': None}, {'fullName': 'Möller, Egert', 'name': 'Egert', 'surname': 'Möller', 'rank': 6, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0009-0004-7288-1553'}, 'provenance': None}}, {'fullName': 'Sanches Pires, Rodrigo', 'name': 'Rodrigo', 'surname': 'Sanches Pires', 'rank': 7, 'pid': {'id': {'sche

In [83]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-2371224906',
  'subjects': ['ID02', 'User Training Proposal', 'UTR-34']},
 {'doi': '10.15151/esrf-es-2313092290',
  'subjects': ['CM01', 'Macromolecular Crystallography', 'MX-2769']},
 {'doi': '10.15151/esrf-es-2002919516',
  'subjects': ['Chemistry', 'ID24-DCM', 'CH-7280', 'x-ray probe']},
 {'doi': '10.15151/esrf-es-1192747525',
  'subjects': ['BM07', 'A07-1-11', '3. Good health']},
 {'doi': '10.15151/esrf-es-1209936304',
  'subjects': ['BM07', 'A07-1-11', '3. Good health']},
 {'doi': '10.15151/esrf-es-1945024757',
  'subjects': ['PaN',
   'ID31',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'IH-MA-563']},
 {'doi': '10.15151/esrf-es-158308267',
  'subjects': ['id30b',
   'id30a1',
   'Macromolecular Crystallography',
   'MX-2073']},
 {'doi': '10.15151/esrf-es-161963533',
  'subjects': ['id30b',
   'id30a1',
   'Macromolecular Crystallography',
   'MX-2073']},
 {'doi': '10.15151/esrf-es-1028462579',
  'subjects': ['PaN',
   'Chemistry',
   'ID12',


In [84]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320',
  'doi': '10.15151/esrf-es-540928320'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525',
  'doi': '10.15151/esrf-es-1192747525'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665',
  'doi': '10.15151/esrf-es-2051950665'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906',
  'doi': '10.15151/esrf-es-2371224906'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051',
  'doi': '10.15151/esrf-es-1896288051'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/161963533',
  'doi': '10.15151/esrf-es-161963533'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/895887653',
  'doi': '10.15151/esrf-es-895887653'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313092290',
  'doi': '10.15151/esrf-es-2313092290'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1945024757',
  'doi': '10.15151/esrf-es-1945024757'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2002919516

In [85]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320',
  'doi': '10.15151/esrf-es-540928320',
  'enriched_subjects': {'BM05',
   'IH-MA-175',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525',
  'doi': '10.15151/esrf-es-1192747525',
  'enriched_subjects': {'3. Good health', 'A07-1-11', 'BM07'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665',
  'doi': '10.15151/esrf-es-2051950665',
  'enriched_subjects': {'BM26',
   'PaN',
   'Particles, Nuclei and Fields',
   'SC-5683',
   'Soft Condensed Matter Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906',
  'doi': '10.15151/esrf-es-2371224906',
  'enriched_subjects': {'ID02', 'UTR-34', 'User Training Proposal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051',
  'doi': '10.15151/esrf-es-1896288051',
  'enriched_subjects': {'CM01',
   'MX-2626',
   'Macromolecular Crystallography',
   'PaN',
   'Particles

In [86]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320',
  'doi': '10.15151/esrf-es-540928320',
  'enriched_subjects': {'BM05',
   'IH-MA-175',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525',
  'doi': '10.15151/esrf-es-1192747525',
  'enriched_subjects': {'3. Good health', 'A07-1-11', 'BM07'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665',
  'doi': '10.15151/esrf-es-2051950665',
  'enriched_subjects': {'BM26',
   'PaN',
   'Particles, Nuclei and Fields',
   'SC-5683',
   'Soft Condensed Matter Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906',
  'doi': '10.15151/esrf-es-2371224906',
  'enriched_subjects': {'ID02', 'UTR-34', 'User Training Proposal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051',
  'doi': '10.15151/esrf-es-1896288051',
  'enriched_subjects': {'CM01',
   'MX-2626',
   'Macromolecular Crystallography',
   'PaN',
   'Particles

In [87]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320',
  'doi': '10.15151/esrf-es-540928320',
  'enriched_subjects': {'BM05',
   'IH-MA-175',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525',
  'doi': '10.15151/esrf-es-1192747525',
  'enriched_subjects': {'3. Good health', 'A07-1-11', 'BM07'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665',
  'doi': '10.15151/esrf-es-2051950665',
  'enriched_subjects': {'BM26',
   'PaN',
   'Particles, Nuclei and Fields',
   'SC-5683',
   'Soft Condensed Matter Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906',
  'doi': '10.15151/esrf-es-2371224906',
  'enriched_subjects': {'ID02', 'UTR-34', 'User Training Proposal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051',
  'doi': '10.15151/esrf-es-1896288051',
  'enriched_subjects': {'CM01',
   'MX-2626',
   'Macromolecular Crystallography',
   'PaN',
   'Particles

In [88]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

150
150


In [89]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [90]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [91]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/540928320',
  'doi': '10.15151/esrf-es-540928320',
  'enriched_subjects': {'BM05',
   'IH-MA-175',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1192747525',
  'doi': '10.15151/esrf-es-1192747525',
  'enriched_subjects': {'3. Good health', 'A07-1-11', 'BM07'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2051950665',
  'doi': '10.15151/esrf-es-2051950665',
  'enriched_subjects': {'BM26',
   'PaN',
   'Particles, Nuclei and Fields',
   'SC-5683',
   'Soft Condensed Matter Science',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2371224906',
  'doi': '10.15151/esrf-es-2371224906',
  'enriched_subjects': {'ID02', 'UTR-34', 'User Training Proposal'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1896288051',
  'doi': '10.15151/esrf-es-1896288051',
  'enriched_subjects': {'CM01',
   'MX-2626',
   'Macromolecular Crystallography',
   'PaN',
   'Particles

In [92]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/540928320</identifier>\n    <datestamp>2021-09-14T06:03:32.145Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-540928320</identifier><creators><creator><creatorName nameType="Personal">Kudakwashe JAKATA</creatorName><givenName>Kudakwashe</givenName><familyName>Jakata</familyName><nameIdentifier nameIdentifierScheme="OR

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [93]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [94]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [95]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [96]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

20


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313092290',
  'doi': '10.15151/esrf-es-2313092290',
  'enriched_subjects': {'Chromatin', 'Signalling'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2264618071',
  'doi': '10.15151/esrf-es-2264618071',
  'enriched_subjects': {'Charge density wave', 'Superconductivity'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2311173392',
  'doi': '10.15151/esrf-es-2311173392',
  'enriched_subjects': {'Mantle (geology)', 'Peridotite', 'Transition zone'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2315949693',
  'doi': '10.15151/esrf-es-2315949693',
  'enriched_subjects': {'Context (archaeology)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2313091200',
  'doi': '10.15151/esrf-es-2313091200',
  'enriched_subjects': {'Catalysis', 'In situ', 'Olefin fiber', 'Syngas'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2255551186',
  'doi': '10.15151/esrf-es-2255551186',
  'enriched_subjects': {'Photonic crystal', 'Ph

In [106]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'},
    {'id': "testid3",
     'doi': '10.1056/NEJMoa2034577'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2', 'doi': '10.23662/fors-ds-686-1', 'enriched_subjects': None},
 {'id': 'testid3',
  'doi': '10.1056/NEJMoa2034577',
  'enriched_subjects': {'Coronavirus disease 2019 (COVID-19)',
   'SARS-CoV-2 and COVID-19 Research',
   'Severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2)'}}]

In [107]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': 'testid3',
  'doi': '10.1056/NEJMoa2034577',
  'enriched_subjects': {'03 medical and health sciences',
   '0302 clinical medicine',
   '3. Good health',
   'Adolescent',
   'Adult',
   'Aged',
   'BNT162 Vaccine',
   'COVID-19',
   'COVID-19 Vaccines',
   'Fatigue',
   'Female',
   'General Medicine',
   'Headache',
   'Humans',
   'Immunization, Secondary',
   'Male',
   'Middle Aged',
   'Original Article',
   'SARS-CoV-2',
   'Single-Blind Method',
   'Treatment Outcome',
   'Vaccines, Synthetic',
   'Young Adult',
   'mRNA Vaccines'}}]

In [108]:
# Combine - add the openAIRE subjects

# TODO: turn into standalone method, lookup by id

for ix in range(len(enriched_results)):
    oaire_subjects = oaire_results[ix]["enriched_subjects"]

    if oaire_subjects:
        current_subjects = enriched_results[ix]['enriched_subjects'] or set()

        # set the current enriched subjects to an union of current and openaire
        enriched_results[ix]['enriched_subjects'] = current_subjects | oaire_subjects

enriched_results 


[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Extraversion and introversion',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Neuroticism',
   'Openness to experience',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': 'testid3',
  'doi': '10.1056/NEJMoa2034577',
  'enriched_subjects': {'03 medical and health sciences',
   '0302 clinical medicine',
   '3. Good health',
   'Adolescent',
   'Adult',
   'Aged',
   'BNT162 Vaccine',
   'COVID-19',
   'COVID-19 Vaccines',
   'Coronavirus disease 2019 (COVID-19)',
   'Fatigue',
   'Female',
   'General Medicine',
   'Headache',
   'Humans',
   'Immunization, Secondary',
   'Male',

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword